# SoundEar3-data — Steg 1: Inläsning och aggregering (batch)

Läser in SE3-CSV-filerna **en i taget** (per dygn) och beräknar:

- **Timstatistik** — antal sampel samt summa och max av 10^(LAF/10) per timme
  (underlag för exakta LAeq/LAFmax i analyssteget)
- **Peaks** — find_peaks på LAF (bullrande passager)
- **Kuvert** — min/medel/max per 10 min (underlag för översiktsplot över hela året)

Resultatet sparas som Parquet-filer i `outpath` och läses av analysnotebooken
`02_analysis.ipynb`. Minnesanvändningen hålls låg eftersom endast en dags data
(~50 MB) finns i minnet åt gången, oavsett hur lång mätperioden är.

Energimedelvärden är additiva: medelvärdet över en timme = (summan av
10^(L/10) för alla sampel) / (antalet sampel). Eftersom filerna är per dygn
och timbins aldrig korsar filgränser blir det uppdelade resultatet **exakt**,
inte en approximation.

Kör om denna notebook när nya datafilern lagts till i `datapath` eller när
justeringsparametrarna (kalibrering, frifält, sommartid) ändrats.


In [1]:
%matplotlib ipympl
#%matplotlib notebook
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import os
import datetime
from scipy.signal import find_peaks

try:
    import pyarrow
except ImportError:
    raise ImportError("pyarrow saknas, installera med: pip install pyarrow")

from se3_common import (set_calibration, save_calibration, cal_for,
                        find_files, read_day, adjust_day)

In [2]:
# Sökvägar
datapath = r'C:\1_Validering\Buller\BG_2025\USBdata'
outpath  = r'C:\1_Validering\Buller\BG_2025\agg'
os.makedirs(outpath, exist_ok=True)

# Inläsningsfönster: filer inom intervallet läses in
startdate = datetime.datetime.strptime('2026-01-01', "%Y-%m-%d").date()
enddate   = datetime.datetime.strptime('2026-12-31', "%Y-%m-%d").date()

# Kalibreringshistorik: (datum, kalibreringskonstant i dB)
# Konstanten ändras linjärt mellan angivna datum. Före första datumet
# gäller det första värdet, efter sista datumet det sista värdet.
# Uppdatera vid varje fältkalibrering och kör om notebooken.
cal_history = [
    ('2026-01-01', 0.0),
]

# Fasadkompensering till frifält (dB)
free_field_comp = 6

# Justering för sommartid: (övergångstidpunkt, shift i timmar)
# Data från och med tidpunkten flyttas 1 h framåt (vår) eller bakåt (höst)
dst_shifts = [
    # ('2026-03-29 03:00:00', 1),
    # ('2026-10-25 03:00:00', -1),
]

# Aktivera kalibreringen (sparas till calibration.json i outpath
# och läses av analysnotebooken)
set_calibration(cal_history)

# Aggregering
agg_param     = 'LAF'     # Parameter som medelvärdesbildas till LAeq
ekvivalenttid = '1h'      # Medelvärdestid för LAeq
envelope_tid  = '10min'   # Upplösning för översiktskuvertet

# Peakdetektering
peak_height     = 80      # dB(A)
peak_prominence = 20      # dB
peak_width      = 8       # sampel (8 sampel = 1 s vid 8 Hz)

In [3]:
# Hitta och sortera datafilerna efter datum
filelist_sorted = find_files(datapath, startdate, enddate)

print(f"{len(filelist_sorted)} filer: {filelist_sorted[0]} .. {filelist_sorted[-1]}")

180 filer: SE3-2009372275-01-01-2026.csv .. SE3-2009372275-29-07-2026.csv


In [ ]:
def hourly_stats(df):
    """Per timme: antal sampel samt summa och max i linjär skala (10^(L/10))."""
    log_lin = 10 ** (df[agg_param] / 10)
    return pd.DataFrame({
        'count':   log_lin.resample(ekvivalenttid, label='left').count(),
        'sum_log': log_lin.resample(ekvivalenttid, label='left').sum(),
        'max_log': log_lin.resample(ekvivalenttid, label='left').max(),
    })


def envelope(df):
    """Min/medel/max av LAF per envelope_tid, för översiktsplot."""
    return df[agg_param].resample(envelope_tid).agg(['min', 'mean', 'max'])


def detect_peaks(df):
    """Peakdetektering av bullrande passager."""
    peaks, properties = find_peaks(
        df[agg_param],
        height=peak_height,
        prominence=peak_prominence,
        width=peak_width,
    )
    return pd.DataFrame({
        'DateTime':    df.index[peaks],
        'Peak_Height': properties['peak_heights'],
        'Prominence':  properties['prominences'],
    }).set_index('DateTime')

In [ ]:
# Bearbeta filerna en i taget — endast en dags data i minnet åt gången
start_time = datetime.datetime.now()

stats_list  = []
env_list    = []
peaks_list  = []

for i, fname in enumerate(filelist_sorted, 1):
    df_day = adjust_day(read_day(datapath, fname), free_field_comp, dst_shifts)
    stats_list.append(hourly_stats(df_day))
    env_list.append(envelope(df_day))
    peaks_list.append(detect_peaks(df_day))
    print(f"[{i}/{len(filelist_sorted)}] {fname}: "
          f"{len(df_day)} sampel, {len(peaks_list[-1])} peaks")
    del df_day

print("Klar på", datetime.datetime.now() - start_time)

In [ ]:
# Sammanställ och spara som Parquet
hourly_stats_df = pd.concat(stats_list)
envelope_df     = pd.concat(env_list)
peaks_df        = pd.concat(peaks_list)

hourly_stats_df.to_parquet(os.path.join(outpath, 'hourly_stats.parquet'))
envelope_df.to_parquet(os.path.join(outpath, 'envelope.parquet'))
peaks_df.to_parquet(os.path.join(outpath, 'peaks.parquet'))
save_calibration(outpath, dst_shifts)

print(f"Timstatistik: {len(hourly_stats_df)} rader")
print(f"Kuvert:       {len(envelope_df)} rader")
print(f"Peaks:        {len(peaks_df)} rader")
print("Kalibrering sparad till calibration.json")

## Kontroll av kalibreringen

Kalibreringskonstanten över perioden — kontroll av att datumen och
nivåerna i `cal_history` blivit rätt.

In [ ]:
fig, ax = plt.subplots(figsize=(9.8, 3))
t = pd.date_range(envelope_df.index[0], envelope_df.index[-1], periods=1000)
ax.plot(t, cal_for(t))
ax.set_ylabel('dB')
ax.set_title('Kalibreringskonstant')
ax.grid()

## Kontrollplot

Kuvertet för hela perioden — en snabb kontroll av att inläsningen blivit rätt.
Detaljerad analys görs i `02_analysis.ipynb`.


In [ ]:
fig, ax = plt.subplots(figsize=(9.8, 5))
ax.fill_between(envelope_df.index, envelope_df['min'], envelope_df['max'],
                alpha=0.3, linewidth=0)
ax.plot(envelope_df.index, envelope_df['mean'], linewidth=0.5)
ax.set_ylabel('dB(A)')
ax.set_title('Ljudnivå (LAF), kontroll av inläst data')
ax.grid()
